# Tracking the Body

We can use computer vision to detection people through poses. We do this using a **Pose Landmarker** model — a neural network that finds **33 points** on a person's body, one for each joint: nose, eyes, shoulders, elbows, wrists, hips, knees, ankles, and more. Drawn together they look like a stick figure laid over the video.

These points are called **landmarks** (or keypoints). They tell you *how* a person is standing or moving, without telling you *who* the person is.

The model runs **on your own computer** in the browser. Nothing is uploaded.

Run the cell below and click **Allow**. Step back so your head and shoulders are in view — the skeleton will follow you for about eight seconds.

In [ ]:
from codetto import cv, graphics
import time

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_pose_detector(camera)

try:
  end = time.time() + 8
  while time.time() < end:
    poses = detector.get_detections()
    canvas.draw_poses(poses)
finally:
  detector.stop()
  camera.stop()

# The Pose Detector

The above three lines of code start the pose detector model, get the detections, and draw the poses on the canvas.

- `cv.start_pose_detector(camera)` starts the pose model and hands you back a **detector object**.
- `detector.get_detections()` gives back a list of **poses**. Each pose is itself a list of 33 landmark dictionaries.
- `canvas.draw_poses(poses)` paints the skeleton over the video — call it every frame.

# Reading a Landmark

Detections are returned as a list of landmarks. Each landmark in the list corresponds with a `cv.POSE` constant, for example `cv.POSE.LEFT_WRIST`, and contains:

| Field | Meaning |
|---|---|
| `x`, `y` | Position on screen, in pixels |
| `z` | Rough depth — how far the point is from the camera |
| `visibility` | How sure the model is that the point is in view, `0.0` to `1.0` |

One thing to remember: `y = 0` is the **top** of the screen and `y` grows **downward**. So a point that is higher up has a *smaller* `y`.

The cell below prints a few key joints once every 60 frames. Run the cell, switch to the console, move closer, lean, step back, and watch the numbers. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_pose_detector(camera)

key_points = [
  ('Nose', cv.POSE.NOSE),
  ('Left shoulder', cv.POSE.LEFT_SHOULDER),
  ('Left wrist', cv.POSE.LEFT_WRIST),
  ('Right wrist', cv.POSE.RIGHT_WRIST),
]

frame = 0
try:
  while True:
    poses = detector.get_detections()
    canvas.draw_poses(poses)
    if frame % 60 == 0:
      if poses:
        pose = poses[0]
        print('--- key landmarks ---')
        for name, index in key_points:
          point = pose[index]
          print(f"  {name}: x={point['x']} y={point['y']} (visible: {point['visibility']:.0%})")
      else:
        print('No pose detected — step back so more of your body is in view.')
    frame += 1
finally:
  detector.stop()
  camera.stop()

# Is an Arm Raised?

Once you can read landmark positions, you can write your own rules on top of them. Here is a simple one: if your **wrist is higher on the screen than your shoulder**, your arm is raised.

Because `y` grows downward, "higher" means the wrist's `y` is a *smaller* number than the shoulder's `y`. That whole idea fits in one comparison: `wrist['y'] < shoulder['y']`.

Run the cell, switch to the console, and raise one or both arms. The status line updates every 30 frames. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_pose_detector(camera)

frame = 0
try:
  while True:
    poses = detector.get_detections()
    canvas.draw_poses(poses)
    if frame % 30 == 0:
      if poses:
        pose = poses[0]
        left_raised = pose[cv.POSE.LEFT_WRIST]['y'] < pose[cv.POSE.LEFT_SHOULDER]['y']
        right_raised = pose[cv.POSE.RIGHT_WRIST]['y'] < pose[cv.POSE.RIGHT_SHOULDER]['y']
        left = 'RAISED' if left_raised else 'at side'
        right = 'RAISED' if right_raised else 'at side'
        print(f'Left arm: {left}  |  Right arm: {right}')
      else:
        print('No pose detected.')
    frame += 1
finally:
  detector.stop()
  camera.stop()

# More Than One Person

By default the Pose Landmarker tracks a single person. Pass `num_poses=2` when you start it and it will track two, returning a longer list from `get_detections()`.

The slider below sets how many people to look for. Change it, re-run the cell, and switch to the console to experiment with different detections. Press **Stop** when you are done.

In [ ]:
from codetto import cv, graphics

NUM_PEOPLE = 1 #@param {type:"slider", min:1, max:2, step:1}

canvas = graphics.canvas()
camera = cv.start_camera(canvas)
detector = cv.start_pose_detector(camera, num_poses=NUM_PEOPLE)

last_count = -1
try:
  while True:
    poses = detector.get_detections()
    canvas.draw_poses(poses)
    if len(poses) != last_count:
      print(f'Tracking {len(poses)} person / people')
      last_count = len(poses)
finally:
  detector.stop()
  camera.stop()

# Check Your Understanding